# Albotherm-01 — Data Cleaning
## Pipeline Step 1 of 4: **Clean** → Merge → EDA → Model

---

### What this notebook does

This is the first step in our four stage analysis pipeline for Albotherm manufacturing data.

Albotherm raw manufacturing and experimental records come from multiple sources — spreadsheets built for day-to-day lab use. This notebook transforms them into clean, consistent, and analysis ready datasets, making it possible to draw meaningful insights and trends.

---

### Source files

| File | What it contains |
|---|---|
| `formulation_database.xlsx` | UV and Core formulation ingredient compositions |
| `outer_formulation.csv` | Outer phase formulation compositions |
| `starting_materials.xlsx` | Batch QC records for UV, Core and Outer batches |
| `axf_manufacturing_run.xlsx` | Production run records (mix conditions, process settings) |
| `durability_results.xlsx` | Humidity chamber test results for each production run |

---

## 1. ADD NEEDED LIBRARY

In [ ]:
import pandas as pd
import numpy as np
import re

## 2. READ FILES

In [ ]:
# formulation dataset has 3 differents including 'UV Formulations', 'Core Formulations', and 'Outer Formulations'
uv          = pd.read_excel('../data/raw/formulation_database.xlsx', sheet_name = 'UV Formulations')
core        = pd.read_excel('../data/raw/formulation_database.xlsx', sheet_name = 'Core Formulations ')
outer       = pd.read_csv('../data/raw//raw/outer_formulation.csv')

# starting materials dataset also has 3 different types of batch, corresponding to 3 formulas.
core_batch  = pd.read_excel('../data/raw/starting_materials.xlsx', sheet_name='Core QC_1')
uv_batch    = pd.read_excel('../data/raw/starting_materials.xlsx', sheet_name='UV QC_1')
outer_batch = pd.read_excel('../data/raw/starting_materials.xlsx', sheet_name='Outer QC')

# axf dataset is the primary production data, including the combination of main materials (UV, core, outer) along with other materials and surrounding conditions.
mix         = pd.read_excel('../data/raw/axf_manufacturing_run.xlsx')

# durability dataset show the quality check result of each axf combination.
result      = pd.read_excel('../data/raw/raw/durability_results.xlsx')

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## 3. UV FORMULATIONS CLEANNING

In [ ]:
print(uv.columns)
print('-'*50)
uv.head(5)

Index(['Tom', 'Unnamed: 1',
       'Welcome to the big formulation database! This page is for UV and next is for core.',
       'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8', 'Unnamed: 9',
       ...
       'Unnamed: 317', 'Unnamed: 318', 'Unnamed: 319', 'Unnamed: 320',
       'Unnamed: 321', 'Unnamed: 322', 'Unnamed: 323', 'Unnamed: 324',
       'Unnamed: 325', 'Unnamed: 326'],
      dtype='object', length=327)
--------------------------------------------------


,Tom,Unnamed: 1,Welcome to the big formulation database! This page is for UV and next is for core.,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 317,Unnamed: 318,Unnamed: 319,Unnamed: 320,Unnamed: 321,Unnamed: 322,Unnamed: 323,Unnamed: 324,Unnamed: 325,Unnamed: 326
0,Kayleigh,NaN,"To find formulations you want, try the filter ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sian,NaN,Raw material amounts are in Wt%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Luke,NaN,Any problems come and find me:) Special thanks...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Izzy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FILTERING,NaN,ESBOA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The first 5 rows and the first 3 columns are actually unimportant as they contain information unnecessary for analysis and will therefore be removed. <br>
<br>
If we observe starting from column 4 (temporarily named 'Unnamed: 3'), which contains the names of formula attributes, then the removal of unnecessary rows will be based on this column. <br>
<br>
Similarly, line 6 contains important information about the names of the experimental samples.

In [ ]:
# drop unecessary rows
uv = uv.dropna(subset=['Unnamed: 3'])

# drop unecessary columns
row_index = 6
uv = uv.dropna(axis=1, subset=[row_index])

uv.head(5)

,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,...,Unnamed: 317,Unnamed: 318,Unnamed: 319,Unnamed: 320,Unnamed: 321,Unnamed: 322,Unnamed: 323,Unnamed: 324,Unnamed: 325,Unnamed: 326
6,Sample,50:50 ESBOA:PEG,THB03-03,THB03-32,AUF010,AUF015,AUF042,AUF069,AUF070,AUF071,...,AUF377,AUF378,AUF379,AUF380,AUF381,AUF382,AUF383,AUF384,AUF385,AUF386
7,Operator,Tom,Tom,Tom,Kayleigh,Kayleigh,Kayleigh,Kayleigh,Kayleigh,Kayleigh,...,Kayleigh,Izzy,Izzy,Kayleigh,Izzy,Izzy,Izzy,Izzy,Izzy,Izzy
8,Made?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,yes,NaN,NaN,yes,yes,yes,NaN,NaN,NaN
9,Date,2023-10-26 00:00:00,2023-10-26 00:00:00,2023-11-03 00:00:00,2022-08-30 00:00:00,2022-11-01 00:00:00,2023-02-08 00:00:00,2023-12-04 00:00:00,2023-12-04 00:00:00,2023-12-06 00:00:00,...,2025-12-10 00:00:00,2025-12-18 00:00:00,2025-12-18 00:00:00,2024-01-06 00:00:00,2026-01-12 00:00:00,2026-01-21 00:00:00,2026-01-21 00:00:00,2026-01-21 00:00:00,2026-01-21 00:00:00,2026-01-21 00:00:00
10,Location of Data,THB Project 2,THB Project 2,THB Project 2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Now we're going to rotate it and reset the index.

In [ ]:
# create a copy of 'uv' and assign it to 'df' so as not to directly modify the original 'uv'.
df = uv.copy()

# get all the first rows and columns as the new label.
row_labels = df.iloc[:, 0]

# the data will be extracted from all rows starting from the second column.
data = df.iloc[:, 1:].copy()

# set the first column as the index for all rows.
data.index = row_labels

# discard items with missing labels.
data = data.loc[data.index.notna()]

# rotate the data frame
uv = data.T.reset_index(drop=True)

# delete column names and delete index names.
uv.columns.name = None
uv.index.name = None

uv.head(5)

,Sample,Operator,Made?,Date,Location of Data,ESBOA,Epoxy:GPTA 75:25,Miramer PU2560,Photocryl DP408NT (40% DPGDA),Miramer U360NT,...,GC1000Z,Total,Viscosity @30 °C,Acrylate density,Flexibility,Cure,"WVP (1= good, 2= mid, 3= poor)",WVP numbers (g/(m2/24hr)) -Pulse data,WVP number - in house data,WVP in-house repeat of pulse films
0,50:50 ESBOA:PEG,Tom,NaN,2023-10-26 00:00:00,THB Project 2,47.5,NaN,NaN,NaN,NaN,...,NaN,100,NaN,3.841131,NaN,NaN,NaN,NaN,NaN,NaN
1,THB03-03,Tom,NaN,2023-10-26 00:00:00,THB Project 2,66.5,NaN,NaN,NaN,NaN,...,NaN,100,NaN,3.254679,NaN,NaN,NaN,NaN,NaN,NaN
2,THB03-32,Tom,NaN,2023-11-03 00:00:00,THB Project 2,NaN,NaN,NaN,NaN,NaN,...,NaN,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,AUF010,Kayleigh,NaN,2022-08-30 00:00:00,NaN,NaN,55.3,NaN,NaN,NaN,...,NaN,100,NaN,Missing data,NaN,NaN,NaN,NaN,NaN,NaN
4,AUF015,Kayleigh,NaN,2022-11-01 00:00:00,NaN,NaN,NaN,49,NaN,NaN,...,NaN,100.0,NaN,2.274751,Good,Mid,3,532,NaN,NaN


In [ ]:
print(uv.columns)

Index(['Sample', 'Operator', 'Made?', 'Date ', ' Location of Data', 'ESBOA',
       'Epoxy:GPTA 75:25', 'Miramer PU2560', 'Photocryl DP408NT (40% DPGDA)',
       'Miramer U360NT',
       ...
       'GC1000Z', 'Total', 'Viscosity @30 °C', 'Acrylate density ',
       'Flexibility', 'Cure', 'WVP (1= good, 2= mid, 3= poor)',
       'WVP numbers (g/(m2/24hr)) -Pulse data', 'WVP number - in house data',
       'WVP in-house repeat of pulse films'],
      dtype='object', length=101)


This section contains columns that are unnecessary in the analysis process, including: 'Operator', 'Made?', 'Date ', ' Location of Data', 'Viscosity @30 °C', 'Acrylate density ', 'Flexibility', 'Cure', 'WVP (1= good, 2= mid, 3= poor)', 'WVP numbers (g/(m2/24hr)) -Pulse data', 'WVP number - in house data', 'WVP in-house repeat of pulse films'.

In [ ]:
uv = uv.drop(columns=['Operator',
                      'Made?',
                      'Date ',
                      ' Location of Data',
                      'Viscosity @30 °C',
                      'Acrylate density ',
                      'Flexibility',
                      'Cure',
                      'WVP (1= good, 2= mid, 3= poor)',
                      'WVP numbers (g/(m2/24hr)) -Pulse data',
                      'WVP number - in house data',
                      'WVP in-house repeat of pulse films'])

There are some blank spaces, all of which indicate that those chemicals are not in the formula, so they will be replaced with 0.

In [ ]:
uv = uv.fillna(0)
uv.head(5)

/tmp/ipykernel_5695/3523612269.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  uv = uv.fillna(0)


,Sample,ESBOA,Epoxy:GPTA 75:25,Miramer PU2560,Photocryl DP408NT (40% DPGDA),Miramer U360NT,Miramer U3195NT,Miramer U3926NT,Miramer U3969NT,Miramer MU9800,...,HS417-1,Gasil HP33C,Span80,Synperonic 91/2.5 LQ,Agitan DF 6422,TEA,GC1100Y,Agisyn 008,GC1000Z,Total
0,50:50 ESBOA:PEG,47.5,0.0,0.0,0.0,0.0,0,0,0,0.0,...,0,0,0,0,0.0,0,0,0.0,0,100.0
1,THB03-03,66.5,0.0,0.0,0.0,0.0,0,0,0,0.0,...,0,0,0,0,0.0,0,0,0.0,0,100.0
2,THB03-32,0.0,0.0,0.0,0.0,0.0,0,0,0,0.0,...,0,0,0,0,0.0,0,0,0.0,0,1.0
3,AUF010,0.0,55.3,0.0,0.0,0.0,0,0,0,0.0,...,0,0,0,0,0.0,0,0,0.0,0,100.0
4,AUF015,0.0,0.0,49.0,0.0,0.0,0,0,0,0.0,...,0,0,0,0,0.0,0,0,0.0,0,100.0


Checking datatype

In [ ]:
for col in uv.columns:
    if col == 'Sample':
        continue
    try:
        uv[col] = uv[col].astype('float64')
    except:
        pass

uv.dtypes[uv.dtypes != 'float64'].to_frame(name='dtype')

,dtype
Sample,object


The file has nothing unusual about its data type.

Upon observation, some experiments in this column lack complete data, resulting in total component percentages ('Total') being less than 100%. <br>
Additionally, some experiments have 'Total' values ​​greater than 100%. <br>
However, these are still used throughout the main production process, so they will be retained for further exploratory and profiling analyses to fully investigate the relationships between the factors.

## 4. CORE FORMULATIONS

In [ ]:
print(core.columns)
print('-'*50)
core.head(5)

Index([' ', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
       'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25',
       'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29',
       'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33',
       'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37',
       'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41',
       'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45',
       'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49',
       'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53',
       'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57'],
      dtype='object')
----------------------------

,,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Filter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Although the data file structure is similar to UV, the important information is contained in two independent columns, and we need to merge them into one column. <br>
<br>
Furthermore, this data file is clearer than the UV file, as the unimportant columns and rows all have one thing in common: they are 100% empty (except for the row containing 'Filter').

In [ ]:
core = core.dropna(how='all')
core = core.dropna(axis=1, how='all')
core.head(5)

,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,...,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57
4,Filter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Sample,NaN,RAD-C-0001,RAD-C-0002,RAD-C-0003,RAD-C-0004,RAD-C-0005,RAD-C-0006,RAD-C-0007,RAD-C-0008,...,RAD-C-0045,RAD-C-0046,RAD-C-0047,RAD-C-0048,RAD-C-0049,RAD-C-0050,RAD-C-0051,RAD-C-0052,RAD-C-0053,RAD-C-0054
7,Operator,NaN,Tom,Tom,Sian,Tom,Tom,Tom,Tom,Tom,...,LAD,THB,THB,LAD,LAD,LAD,LAD,LAD,LAD,LAD
8,Date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2026-01-15 00:00:00,2026-01-15 00:00:00,2026-01-16 00:00:00,2026-01-16 00:00:00,2026-01-20 00:00:00,2026-01-21 00:00:00,2026-01-23 00:00:00,2026-01-28 00:00:00,2026-01-29 00:00:00,2026-01-29 00:00:00
9,OG name,NaN,OG core,94/6 (25%),94/6 (20%),95/5 (25%),95/5 (20%),96/4 (25%),96/4 (20%),93/7 (20%),...,Hydrolysed KE 1.25kg 25% batch,NaN,NaN,"OG with 1,5-PDO",20% HPC in 1:3 water:PEG200,Endpoint hyd HPC w/5.4% NaCl,Nisso 100k glycerol core,OG core with Nisso20% 100k,76 Nisso20% 100k / 17 / 7,76 Nisso30% 100k / 17 / 7


Rotate it and reset the index.

In [ ]:
df = core.copy()

row_labels = df.iloc[:, :2].bfill(axis=1).iloc[:, 0]

data = df.iloc[:, 2:].copy()

data.index = row_labels

data = data.loc[data.index.notna()]

core = data.T.reset_index(drop=True)

core.columns.name = None

core.index.name = None

core.head(5)

,Filter,Sample,Operator,Date,OG name,Klucel E,Klucel ELF,Nisso 40 K,Nisso 100 K,Hyd HPC unknown MW,...,Klucel E,Klucel ELF,Nisso 40 K,Nisso 100 K,Hyd HPC unknown MW,Hyd HPC 7k,Hyd HPC 15k,Hyd HPC 20k,Hyd KE HPC 9k,Hyd KE HPC 5k
0,NaN,RAD-C-0001,Tom,NaN,OG core,13.2,NaN,NaN,NaN,NaN,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,RAD-C-0002,Tom,NaN,94/6 (25%),23.5,NaN,NaN,NaN,NaN,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,RAD-C-0003,Sian,NaN,94/6 (20%),18.8,NaN,NaN,NaN,NaN,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,RAD-C-0004,Tom,NaN,95/5 (25%),23.75,NaN,NaN,NaN,NaN,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,RAD-C-0005,Tom,NaN,95/5 (20%),19,NaN,NaN,NaN,NaN,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(core.columns)

Index(['Filter', 'Sample', 'Operator', 'Date ', 'OG name', 'Klucel E',
       'Klucel ELF ', 'Nisso 40 K ', 'Nisso 100 K ', 'Hyd HPC unknown MW',
       'Hyd HPC 7k', 'Hyd HPC 15k', 'Hyd HPC 20k', 'Hyd KE HPC 9k',
       'Hyd KE HPC 5k', 'NaCl', 'CaCl2', 'PDO', 'Glycerol ', '1,5-PDO', 'MPG ',
       'C12E6', 'Water ', 'Go Water Solution (0.3%?)', 'PEG550DMA ', 'PEG200',
       'Total', 'Pass/ fail (homogeneous) ', 'Viscosity @25', 'LCST ',
       'Refractive index (%Brix)', 'DT ',
       'Concentration of HPC stock (wt% HPC)', 'Amount of stock added (wt%) ',
       'Klucel E', 'Klucel ELF ', 'Nisso 40 K ', 'Nisso 100 K ',
       'Hyd HPC unknown MW', 'Hyd HPC 7k', 'Hyd HPC 15k', 'Hyd HPC 20k',
       'Hyd KE HPC 9k', 'Hyd KE HPC 5k'],
      dtype='object')


This section contains columns that are unnecessary in the analysis process, including: 'Filter', 'Operator', 'Date ', 'OG name', 'Pass/ fail (homogeneous) ', 'Viscosity @25', 'LCST ', 'Refractive index (%Brix)', 'DT ', 'Concentration of HPC stock (wt% HPC)', 'Amount of stock added (wt%) ', 'Klucel E', 'Klucel ELF ', 'Nisso 40 K ', 'Nisso 100 K ', 'Hyd HPC unknown MW', 'Hyd HPC 7k', 'Hyd HPC 15k', 'Hyd HPC 20k', 'Hyd KE HPC 9k', 'Hyd KE HPC 5k'

In [ ]:
core = core.drop(columns=['Filter','Operator',
                          'Date ',
                          'OG name',
                          'Pass/ fail (homogeneous) ',
                          'Viscosity @25',
                          'LCST ',
                          'Refractive index (%Brix)',
                          'DT ',
                          'Concentration of HPC stock (wt% HPC)',
                          'Amount of stock added (wt%) '])

# since the last 10 columns have the same names as the columns in the formula, iloc will be used to select all columns except the last 10.
core = core.iloc[:, :-10]

core.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Sample                     54 non-null     object
 1   Klucel E                   29 non-null     object
 2   Klucel ELF                 3 non-null      object
 3   Nisso 40 K                 4 non-null      object
 4   Nisso 100 K                4 non-null      object
 5   Hyd HPC unknown MW         2 non-null      object
 6   Hyd HPC 7k                 2 non-null      object
 7   Hyd HPC 15k                4 non-null      object
 8   Hyd HPC 20k                5 non-null      object
 9   Hyd KE HPC 9k              2 non-null      object
 10  Hyd KE HPC 5k              0 non-null      object
 11  NaCl                       51 non-null     object
 12  CaCl2                      1 non-null      object
 13  PDO                        16 non-null     object
 14  Glycerol    

There are some blank spaces, all of which indicate that those chemicals are not in the formula, so they will be replaced with 0.

In [ ]:
core = core.fillna(0)
core.head(5)

/tmp/ipykernel_5695/2952262540.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  core = core.fillna(0)


,Sample,Klucel E,Klucel ELF,Nisso 40 K,Nisso 100 K,Hyd HPC unknown MW,Hyd HPC 7k,Hyd HPC 15k,Hyd HPC 20k,Hyd KE HPC 9k,...,PDO,Glycerol,"1,5-PDO",MPG,C12E6,Water,Go Water Solution (0.3%?),PEG550DMA,PEG200,Total
0,RAD-C-0001,13.20,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,...,27.0,0.0,0,0.0,0.0,52.8,0.0,0.0,0,100.0
1,RAD-C-0002,23.50,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,70.5,0.0,0.0,0,100.0
2,RAD-C-0003,18.80,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,75.2,0.0,0.0,0,100.0
3,RAD-C-0004,23.75,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,71.25,0.0,0.0,0,100.0
4,RAD-C-0005,19.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,76,0.0,0.0,0,100.0


Adjust data type

In [ ]:
for col in core.columns:
    if col == 'Sample':
        continue
    try:
        core[col] = core[col].astype('float64')
    except:
        pass

core.dtypes.to_frame(name='dtype')

,dtype
Sample,object
Klucel E,float64
Klucel ELF,float64
Nisso 40 K,float64
Nisso 100 K,float64
Hyd HPC unknown MW,float64
Hyd HPC 7k,float64
Hyd HPC 15k,object
Hyd HPC 20k,float64
Hyd KE HPC 9k,float64


Columns 'Hyd HPC 15k' and 'Water ' cannot be converted to float, so there may be incorrect values.

In [ ]:
print(core['Hyd HPC 15k'].unique())
print('-'*50)
print(core['Water '].unique())

[0 19.050325 24.552 40 'x']
--------------------------------------------------
[52.8 70.5 75.2 71.25 76 72 76.8 69.75 49.5 52.5 52.81399999999999 73.728
 51.767999999999994 52.28 52.536 52.74400000000001 78.96 55.51200000000001
 77.868 75.609675 72.16799999999999 68.448 56.5 'y' 41.25 57.6 51.2 75.952
 56.856 76.3 52 52.2 0 62.4 51 70.92 49.8 50.4 15 75.82 60.8 53.2]


The data in these two columns contains text values, so they will be replaced with NaN because the information related to this formula is still needed for evaluating the relationship between the factors.

In [ ]:
core[['Hyd HPC 15k', 'Water ']] = core[['Hyd HPC 15k', 'Water ']].replace(['x', 'y'], np.nan)

/tmp/ipykernel_5695/751194076.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  core[['Hyd HPC 15k', 'Water ']] = core[['Hyd HPC 15k', 'Water ']].replace(['x', 'y'], np.nan)


Re-adjust datatype

In [ ]:
for col in core.columns:
    if col == 'Sample':
        continue
    try:
        core[col] = core[col].astype('float64')
    except:
        pass

core.dtypes.to_frame(name='dtype')

,dtype
Sample,object
Klucel E,float64
Klucel ELF,float64
Nisso 40 K,float64
Nisso 100 K,float64
Hyd HPC unknown MW,float64
Hyd HPC 7k,float64
Hyd HPC 15k,float64
Hyd HPC 20k,float64
Hyd KE HPC 9k,float64


Similarly to UV formulation, some experiments have a 'Total' different from 100%, but these are still kept to preserve information.

## 5. OUTER FORMULATION

In [ ]:
print(outer.columns)
print('-'*50)
outer.head(5)

Index([' ', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
       'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25',
       'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29',
       'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33',
       'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37',
       'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41',
       'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45',
       'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49',
       'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53',
       'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57'],
      dtype='object')
----------------------------

,,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Filter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Its structure is exactly the same as a 'Core' file, so the operation is the same.

In [ ]:
outer = outer.dropna(how='all')
outer = outer.dropna(axis=1, how='all')
outer.head(5)

,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
4,Filter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Sample,NaN,2.25/38G/7,4.5/14.7P/4,2.25/38P/7,2.25/38M/7,4.5/14.7M/4,4.5/19M/6,2.25/27P/11S/7
7,Operator,NaN,Kayleigh,Luke,Kayleigh,Luke,Luke,Luke,Luke
8,Date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,OG name,NaN,AXF-1 outer,96/4 Outer,PDO Outer,MPG Outer,96/4 Outer,94/6 Outer,Sugar Outer


In [ ]:
df = outer.copy()

row_labels = df.iloc[:, :2].bfill(axis=1).iloc[:, 0]

data = df.iloc[:, 2:].copy()

data.index = row_labels

data = data.loc[data.index.notna()]

outer = data.T.reset_index(drop=True)

outer.columns.name = None

outer.index.name = None

outer

,Filter,Sample,Operator,Date,OG name,NaCl,Glycerol,PDO,MPG,Sugar,PVA (10% in water),Water
0,NaN,2.25/38G/7,Kayleigh,NaN,AXF-1 outer,7,38,NaN,NaN,NaN,2.25,52.75
1,NaN,4.5/14.7P/4,Luke,NaN,96/4 Outer,4,NaN,14.7,NaN,NaN,4.5,76.8
2,NaN,2.25/38P/7,Kayleigh,NaN,PDO Outer,7,NaN,38,NaN,NaN,2.25,52.75
3,NaN,2.25/38M/7,Luke,NaN,MPG Outer,7,NaN,NaN,38,NaN,2.25,52.75
4,NaN,4.5/14.7M/4,Luke,NaN,96/4 Outer,4,NaN,NaN,14.7,NaN,4.5,76.8
5,NaN,4.5/19M/6,Luke,NaN,94/6 Outer,6,NaN,NaN,19,NaN,4.5,70.5
6,NaN,2.25/27P/11S/7,Luke,NaN,Sugar Outer,7,NaN,27,NaN,11,2.25,52.75


In [ ]:
outer.columns

Index(['Filter', 'Sample', 'Operator', 'Date ', 'OG name', 'NaCl', 'Glycerol',
       'PDO', 'MPG', 'Sugar', 'PVA (10% in water)', 'Water'],
      dtype='object')

In [ ]:
outer = outer.drop(columns=['Filter','Date ', 'Operator', 'OG name'])
outer = outer.fillna(0)
outer

,Sample,NaCl,Glycerol,PDO,MPG,Sugar,PVA (10% in water),Water
0,2.25/38G/7,7,38,0,0,0,2.25,52.75
1,4.5/14.7P/4,4,0,14.7,0,0,4.5,76.8
2,2.25/38P/7,7,0,38,0,0,2.25,52.75
3,2.25/38M/7,7,0,0,38,0,2.25,52.75
4,4.5/14.7M/4,4,0,0,14.7,0,4.5,76.8
5,4.5/19M/6,6,0,0,19,0,4.5,70.5
6,2.25/27P/11S/7,7,0,27,0,11,2.25,52.75


In [ ]:
outer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Sample              7 non-null      object
 1   NaCl                7 non-null      object
 2   Glycerol            7 non-null      object
 3   PDO                 7 non-null      object
 4   MPG                 7 non-null      object
 5   Sugar               7 non-null      object
 6   PVA (10% in water)  7 non-null      object
 7   Water               7 non-null      object
dtypes: object(8)
memory usage: 580.0+ bytes


In [ ]:
for col in outer.columns:
    if col == 'Sample':
        continue
    try:
        outer[col] = outer[col].astype('float64')
    except:
        pass

outer.dtypes.to_frame(name='dtype')

,dtype
Sample,object
NaCl,float64
Glycerol,float64
PDO,float64
MPG,float64
Sugar,float64
PVA (10% in water),float64
Water,float64


Similarly to UV formulation, some experiments have a 'Total' different from 100%, but these are still kept to preserve information.

## 6. CORE BATCH

In [ ]:
core_batch.head(5)

,Records of QC tests carried out on batches of core made,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Old name,Formulation,Batch Number,Date,Batch Quantity (kg),Viscosity (cP) @ RT 5 RPM,Visco (s DIN4),Viscosity (cP) @ RT 20 RPM,Viscosity temperature (°C),Starting LCST temperature (°C),Finish LCST temperature (°C),Osmolarity @ 0.25x,QC operator initials,Pass/Fail,Comments
2,KE core 02-12 (AXF-1),RAD-C-0011,BNC00001,2024-03-25 00:00:00,2,NaN,308,NaN,NaN,18,18,1278,KLM,Pass,NaN
3,KE core 02-12 (Telos),RAD-C-0011,BNC00002,2024-04-02 00:00:00,0.774,NaN,53,NaN,NaN,13,NaN,1800,KLM,Pass,NaN
4,KE core 02-12T,RAD-C-0011,BNC00003,2024-04-09 00:00:00,0.89,NaN,79.27,NaN,NaN,14.5,NaN,1745,LAD,Pass,"Osmolarity high, +0.5% salt"


In [ ]:
new_header = core_batch.iloc[1]

core_batch = core_batch.iloc[2:].copy()

core_batch.columns = new_header

core_batch = core_batch.reset_index(drop=True)

core_batch.columns.name = None

core_batch.head(5)

,Old name,Formulation,Batch Number,Date,Batch Quantity (kg),Viscosity (cP) @ RT 5 RPM,Visco (s DIN4),Viscosity (cP) @ RT 20 RPM,Viscosity temperature (°C),Starting LCST temperature (°C),Finish LCST temperature (°C),Osmolarity @ 0.25x,QC operator initials,Pass/Fail,Comments
0,KE core 02-12 (AXF-1),RAD-C-0011,BNC00001,2024-03-25 00:00:00,2,NaN,308,NaN,NaN,18,18,1278,KLM,Pass,NaN
1,KE core 02-12 (Telos),RAD-C-0011,BNC00002,2024-04-02 00:00:00,0.774,NaN,53,NaN,NaN,13,NaN,1800,KLM,Pass,NaN
2,KE core 02-12T,RAD-C-0011,BNC00003,2024-04-09 00:00:00,0.89,NaN,79.27,NaN,NaN,14.5,NaN,1745,LAD,Pass,"Osmolarity high, +0.5% salt"
3,KE core 02-12T,RAD-C-0011,BNC00004,2024-04-16 00:00:00,1.06,NaN,62,NaN,NaN,15,NaN,1868,LAD,Pass,NaN
4,KE core 02-12A,RAD-C-0011,BNC00005,2024-04-23 00:00:00,0.8,NaN,182,NaN,NaN,15.4,18.2,1875,THB,Pass,NaN


In [ ]:
core_batch = core_batch.drop(columns=['Old name','Date', 'Comments'])
core_batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 12 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Formulation                      110 non-null    object
 1   Batch Number                     117 non-null    object
 2   Batch Quantity (kg)              115 non-null    object
 3   Viscosity (cP) @ RT 5 RPM        79 non-null     object
 4   Visco (s DIN4)                   12 non-null     object
 5   Viscosity (cP) @ RT 20 RPM       5 non-null      object
 6   Viscosity temperature (°C)       43 non-null     object
 7   Starting LCST temperature (°C)   38 non-null     object
 8   Finish LCST temperature (°C)     9 non-null      object
 9   Osmolarity @ 0.25x               32 non-null     object
 10  QC operator initials             43 non-null     object
 11  Pass/Fail                        12 non-null     object
dtypes: object(12)
memory usage: 11.1+ KB

## 7. UV BATCH

In [ ]:
uv_batch.head(5)

,Formulation,Batch Number,Date made,Batch Quantity (kg),Viscosity @ RT RPM,Temp (°C) at visco measurement,Cure,QC operator initials,QC addition,Pass/Fail,Comments
0,AUF069,BNU00001,2024-04-26 00:00:00,1.0,185.0,NaN,Y,KLM,NaN,Pass,NaN
1,AUF069,BNU00002,2024-04-30 00:00:00,5.0,206.0,NaN,Y,KLM,NaN,Pass,NaN
2,AUF069,BNU00003,2024-05-08 00:00:00,1.0,230.0,NaN,Y,KLM,0.7% TPGDA,Pass,NaN
3,AUF069,BNU00004,2024-05-16 00:00:00,5.5,242.0,NaN,Y,KLM,NaN,NaN,NaN
4,Adjusted UV for Telos from AXF,BNU00005,2024-05-31 00:00:00,1.0,215.0,NaN,Y,KLM,NaN,Pass,NaN


In [ ]:
uv_batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Formulation                      177 non-null    object 
 1   Batch Number                     177 non-null    object 
 2   Date made                        169 non-null    object 
 3   Batch Quantity (kg)              177 non-null    float64
 4   Viscosity @ RT RPM               164 non-null    float64
 5   Temp (°C) at visco measurement   84 non-null     float64
 6   Cure                             127 non-null    object 
 7   QC operator initials             129 non-null    object 
 8   QC addition                      1 non-null      object 
 9   Pass/Fail                        12 non-null     object 
 10  Comments                         20 non-null     object 
dtypes: float64(3), object(8)
memory usage: 15.3+ KB


In [ ]:
uv_batch = uv_batch.drop(columns=['Date made ', 'Comments'])
uv_batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Formulation                      177 non-null    object 
 1   Batch Number                     177 non-null    object 
 2   Batch Quantity (kg)              177 non-null    float64
 3   Viscosity @ RT RPM               164 non-null    float64
 4   Temp (°C) at visco measurement   84 non-null     float64
 5   Cure                             127 non-null    object 
 6   QC operator initials             129 non-null    object 
 7   QC addition                      1 non-null      object 
 8   Pass/Fail                        12 non-null     object 
dtypes: float64(3), object(6)
memory usage: 12.6+ KB


## 8. OUTER BATCH

In [ ]:
outer_batch.head(5)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,0.838889,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Formulation,Batch Number,Date made,Batch Quantity (kg),Viscosity @ RT (t = 10mins) 100 RPM,Osmolarity @ 0.25x,Temp @ visco (°C),QC operator initials,QC addition,Pass/Fail,Comments
2,NaN,Telos outer (rusty PVA),BNO00001,2024-03-27 00:00:00,20,20 cp/ 15s (DIN4),1022,NaN,KLM,NaN,Pass,NaN
3,NaN,AXF_1 outer,BNO00002,2024-03-28 00:00:00,20,8,1000,NaN,KLM,NaN,Fail,NaN
4,NaN,Telos outer,BNO00003,2024-04-02 00:00:00,5,20.8,1034,NaN,KLM,NaN,Pass,NaN


In [ ]:
new_head = outer_batch.iloc[1]

outer_batch =  outer_batch.iloc[2:].copy()

outer_batch.columns = new_head

outer_batch = outer_batch.reset_index(drop=True)

outer_batch.columns.name = None

outer_batch.head(5)

,NaN,Formulation,Batch Number,Date made,Batch Quantity (kg),Viscosity @ RT (t = 10mins) 100 RPM,Osmolarity @ 0.25x,Temp @ visco (°C),QC operator initials,QC addition,Pass/Fail,Comments
0,NaN,Telos outer (rusty PVA),BNO00001,2024-03-27 00:00:00,20,20 cp/ 15s (DIN4),1022,NaN,KLM,NaN,Pass,NaN
1,NaN,AXF_1 outer,BNO00002,2024-03-28 00:00:00,20,8,1000,NaN,KLM,NaN,Fail,NaN
2,NaN,Telos outer,BNO00003,2024-04-02 00:00:00,5,20.8,1034,NaN,KLM,NaN,Pass,NaN
3,NaN,AXF_1 outer,BNO00004,2024-04-02 00:00:00,20,12s (DIN4),1026,NaN,KLM/LAD,NaN,Pass,NaN
4,NaN,Telos outer,BNO00005,2024-04-03 00:00:00,2,14s (DIN4),1148.666667,NaN,KLM/LAD,NaN,Pass,NaN


In [ ]:
outer_batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   nan                                  0 non-null      float64
 1   Formulation                          255 non-null    object 
 2   Batch Number                         254 non-null    object 
 3   Date made                            255 non-null    object 
 4   Batch Quantity (kg)                  252 non-null    object 
 5   Viscosity @ RT (t = 10mins) 100 RPM  232 non-null    object 
 6   Osmolarity @ 0.25x                   57 non-null     object 
 7   Temp @ visco (°C)                    55 non-null     object 
 8   QC operator initials                 221 non-null    object 
 9   QC addition                          1 non-null      object 
 10  Pass/Fail                            30 non-null     object 
 11  Comments                        

In [ ]:
outer_batch = outer_batch.drop(columns=[outer_batch.columns[0], 'Date made ', 'Comments'])
outer_batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   Formulation                          255 non-null    object
 1   Batch Number                         254 non-null    object
 2   Batch Quantity (kg)                  252 non-null    object
 3   Viscosity @ RT (t = 10mins) 100 RPM  232 non-null    object
 4   Osmolarity @ 0.25x                   57 non-null     object
 5   Temp @ visco (°C)                    55 non-null     object
 6   QC operator initials                 221 non-null    object
 7   QC addition                          1 non-null      object
 8   Pass/Fail                            30 non-null     object
dtypes: object(9)
memory usage: 18.1+ KB


## 9. AXF MANUFACTORING

In [ ]:
mix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 32 columns):
 #   Column                                               Non-Null Count  Dtype         
---  ------                                               --------------  -----         
 0   AXF Number                                           628 non-null    object        
 1   Date of experiment                                   628 non-null    datetime64[ns]
 2   Aims & Hypothesis                                    627 non-null    object        
 3   Core used                                            628 non-null    object        
 4   Core Formulation                                     628 non-null    object        
 5   Core Viscosity (cP)                                  628 non-null    int64         
 6   Amount of Core Used (g)                              625 non-null    float64       
 7   UV used                                              628 non-null    object        
 8   

In [ ]:
for col in mix.columns:
    if col in ['AXF Number', 'Date of experiment', 'Aims & Hypothesis']:
        continue
    try:
        mix[col] = mix[col].astype('float64')
    except:
        pass

mix.dtypes.to_frame(name='dtype')

,dtype
AXF Number,object
Date of experiment,datetime64[ns]
Aims & Hypothesis,object
Core used,object
Core Formulation,object
Core Viscosity (cP),float64
Amount of Core Used (g),float64
UV used,object
UV formulation,object
UV viscosity (cP),float64


Observation:

The mix table shows that most batch IDs follow the standard `AXF####`
format, but a number of early and experimental runs were logged with texts — for example
`AXF0050_large`, `AXF0033_H2O`, `AXF0033_glycerol`.

For merging purposes, we only need the main AXF number. We extract it from every ID andproduce a consistent `batch_key`.  The original `AXF Number` column is kept unchanged so no information is lost.

In [ ]:
print(" Non-standard IDs in mix ")
mask_mix = ~mix['AXF Number'].str.match(r'^AXF\d{4}$', na=False)
print(mix.loc[mask_mix, 'AXF Number'].head(10).tolist())

 Non-standard IDs in mix 
['AXF0032_3 UV', 'AXF0033_core', 'AXF0033_H2O', 'AXF0033_outer', 'AXF0033_0.9x outer', 'AXF0033_glycerol', 'AXF0047_large', 'AXF0047_small', 'AXF0050_large', 'AXF0050_small']


In [ ]:
def normalise_axf(val):
    val = str(val).strip().upper()
    m = re.match(r'AXF0*(\d+)', val)
    return f"AXF{int(m.group(1)):04d}" if m else val

mix['batch_key'] = mix['AXF Number'].apply(normalise_axf)

print(f"\n mix after normalisation")
print(f"Total rows         : {len(mix)}")
print(f"Unique batch_key   : {mix['batch_key'].nunique()}")
print(f"Non-standard fixed : {mask_mix.sum()}")


 mix after normalisation
Total rows         : 628
Unique batch_key   : 612
Non-standard fixed : 21


## 10. DURABILITY

In [ ]:
result.info()
result.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 679 entries, 0 to 678
Data columns (total 28 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Batch number                            679 non-null    object 
 1   UV formulation                          679 non-null    object 
 2   Core formulation                        679 non-null    object 
 3   Date durability set up                  58 non-null     object 
 4   manufacturing notes                     628 non-null    object 
 5   On/Off at RT?                           110 non-null    object 
 6   Coating compatibility?                  172 non-null    object 
 7   Bulk coating compability?               171 non-null    object 
 8   % dry after 24h in humidity             620 non-null    object 
 9   % dry after 2 days in humidity chamber  563 non-null    object 
 10  Unnamed: 10                             646 non-null    float6

,Batch number,UV formulation,Core formulation,Date durability set up,manufacturing notes,On/Off at RT?,Coating compatibility?,Bulk coating compability?,% dry after 24h in humidity,% dry after 2 days in humidity chamber,...,Column27,Column28,Column29,Column30,Column31,Column32,Column33,Column34,Column35,batch_key
0,AXF0340,AUF75,RAD-C-0001,NaN,50m tubing,NaN,NaN,NaN,5,"10 air dry, 100 polymer dry",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AXF0340
1,AXF0196,AUF96,RAD-C-0001,NaN,NaN,off,NaN,NaN,50,-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AXF0196
2,AXF0495,AUF110,RAD-C-0002,NaN,cure rig 4.0 25% power 8mm insert,NaN,NaN,NaN,50,-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AXF0495
3,AXF0494,AUF110,RAD-C-0002,NaN,cure rig 4.0 10% power 8mm insert,NaN,NaN,NaN,60,-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AXF0494
4,AXF0496,AUF110,RAD-C-0002,NaN,cure rig 4.0 10% power,NaN,NaN,NaN,70,-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AXF0496


Observation

The result table records humidity chamber test outcomes for each production run.
It had three issues that needed resolving before export:

**1. Empty columns** — `Column23` through `Column35` and `Unnamed: 10` were carried over
from the original Excel layout but contain no data. These are removed.

**2. Non-standard batch IDs** — Some IDs are missing zero (`AXF176` instead of
`AXF0176`), have decimal suffixes (`AXF0542.1`), or contain text labels like `AXF0045 (LARGE)`.
We apply the same normalisation used in mix to produce a consistent `batch_key`.

**3 the two dryness columns** are still recorded as `dtype: object` —
meaning pandas is treating them as text, not numbers. We convert them to numeric here, so
`dryness_24h` and `dryness_48h` are clean `float64` columns ready for analysis.


In [ ]:
result = result.drop(columns=['Unnamed: 10', 'Column23', 'Column25', 'Column26', 'Column27', 'Column28', 'Column29', 'Column30', 'Column31', 'Column32', 'Column33', 'Column34', 'Column35'])
result.head(5)

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Batch number,UV formulation,Core formulation,Date durability set up,manufacturing notes,On/Off at RT?,Coating compatibility?,Bulk coating compability?,% dry after 24h in humidity,% dry after 2 days in humidity chamber,Do caps look on or off at 24hrs?,Do caps turn off and/or on? at 24hrs,QC Notes,QC Pass/Fail?,batch_key
0,AXF0340,AUF75,RAD-C-0001,NaN,50m tubing,NaN,NaN,NaN,5,"10 air dry, 100 polymer dry",NaN,NaN,used the same coating sample,NaN,AXF0340
1,AXF0196,AUF96,RAD-C-0001,NaN,NaN,off,NaN,NaN,50,-,NaN,NaN,everything here onwards used 5 meter tubing on...,NaN,AXF0196
2,AXF0495,AUF110,RAD-C-0002,NaN,cure rig 4.0 25% power 8mm insert,NaN,NaN,NaN,50,-,NaN,NaN,NaN,NaN,AXF0495
3,AXF0494,AUF110,RAD-C-0002,NaN,cure rig 4.0 10% power 8mm insert,NaN,NaN,NaN,60,-,NaN,NaN,NaN,NaN,AXF0494
4,AXF0496,AUF110,RAD-C-0002,NaN,cure rig 4.0 10% power,NaN,NaN,NaN,70,-,NaN,NaN,NaN,NaN,AXF0496


In [ ]:
print("Non-standard IDs in result")
mask_res = ~result['Batch number'].str.match(r'^AXF\d{4}$', na=False)
print(result.loc[mask_res, 'Batch number'].tolist())

result['batch_key'] = result['Batch number'].apply(normalise_axf)

print(f"\n Result after normalisation")
print(f"Total rows         : {len(result)}")
print(f"Unique batch_key   : {result['batch_key'].nunique()}")
print(f"Non-standard fixed : {mask_res.sum()}")

Non-standard IDs in result
['AXF0494 ', 'AXF0496 ', 'AXF0498 ', 'AXF0495 ', 'AXF0135 ', 'AXF0152 ', 'AXF0504 ', 'AXF0546 ', 'AXF0541 ', 'AXF0543 ', 'AXF0545 ', 'AXF0605 ', 'AXF0503 ', 'AXF0606 ', 'AXF0608 ', 'AXF176 ', 'AXF0527 ', 'AXF0166 ', 'AXF0167 ', 'AXF0168 ', 'AXF0085 ', 'AXF0526 ', 'AXF0535 ', 'AXF0528 ', 'AXF0527 ', 'AXF0133 ', 'AXF0546 ', 'AXF0510 ', 'AXF0545 ', 'AXF0545.1 ', 'AXF0508 ', 'AXF0540 ', 'AXF0505 ', 'AXF0506 ', 'AXF0507 ', 'AXF0490 ', 'AXF0491 ', 'AXF0492 ', 'AXF0493 ', 'AXF0493 ', 'AXF0494 ', 'AXF0496 ', 'AXF0497 ', 'AXF0498 ', 'AXF0500 ', 'AXF0501 ', 'AXF0502 ', 'AXF0503 ', 'AXF0054 ', 'AXF0143 ', 'AXF0144 ', 'AXF0145 ', 'AXF0146 ', 'AXF0147 ', 'AXF0148 ', 'AXF0149 ', 'AXF0150 ', 'AXF0151 ', 'AXF0153 ', 'AXF0154 ', 'AXF0155 ', 'AXF0156 ', 'AXF0157 ', 'AXF0158 ', 'AXF0159 ', 'AXF0160 ', 'AXF0161 ', 'AXF0162 ', 'AXF0163 ', 'AXF0490 ', 'AXF0491 ', 'AXF0492 ', 'AXF0499 ', 'AXF0045 (large)', 'AXF0045 - 7 days', 'AXF0058 ', 'AXF0168 ', 'AXF0546 ', 'AXF0051 ', 'AXF0542

In [ ]:
result['% dry after 24h in humidity'] = pd.to_numeric(
    result['% dry after 24h in humidity'], errors='coerce'
)
result['% dry after 2 days in humidity chamber'] = pd.to_numeric(
    result['% dry after 2 days in humidity chamber'], errors='coerce'
)

print(f"\n% dry after 24h dtype : {result['% dry after 24h in humidity'].dtype}")
print(f"% dry after 2 days dtype : {result['% dry after 2 days in humidity chamber'].dtype}")


% dry after 24h dtype : float64
% dry after 2 days dtype : float64


In [ ]:
print("All files loaded\n")
for name, df in [("uv", uv), ("core", core), ("outer", outer),
                 ("core_batch", core_batch), ("uv_batch", uv_batch),
                 ("outer_batch", outer_batch), ("mix", mix), ("result", result)]:
    print(f"  {name:15s}: {df.shape}")

All files loaded

  uv             : (323, 89)
  core           : (54, 23)
  outer          : (7, 8)
  core_batch     : (117, 12)
  uv_batch       : (177, 9)
  outer_batch    : (255, 9)
  mix            : (628, 33)
  result         : (679, 15)


## SUMMARY

This Notebook transformed 8 raw source files into clean, analysis-ready datasets.
Formulation and batch tables were restructured from wide Excel layouts into tidy row per sample format. The two primary tables used for modelling — `mix` (628 production runs) and `result`
(679 durability test records) — have been standardised with a shared `batch_key` column for reliable merging, and the dryness measurement columns in `result` have been converted from mixed text to numeric values. All 8 cleaned files are now ready for the
next step: **albotherm_02_merge**.

## EXPORT DATAFRAMES

In [ ]:
uv.to_csv('../data/processed/uv_cleaned.csv', index = False)
core.to_csv('../data/processed/core_cleaned.csv', index = False)
outer.to_csv('/content/drive/MyDrive/albothrm/processed/outer_cleaned.csv', index = False)

uv_batch.to_csv('../data/processed/uv_batch_cleaned.csv', index = False)
core_batch.to_csv('../data/processed/core_batch_cleaned.csv', index = False)
outer_batch.to_csv('../data/processed/outer_batch_cleaned.csv', index = False)

mix.to_csv('../data/processed/mix_cleaned.csv', index = False)
result.to_csv('../data/processed/result_cleaned.csv', index = False)